In [31]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
import joblib
from pathlib import Path




In [14]:
df = pd.read_csv("../data/transactions_clean.csv")
# df[["description", "category"]].head(10)
X = df["description"]
Y = df["category"]

# print(X.head(3))
# print(Y.head(3))
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

print("Study (train):", len(X_train))
print("Exam (test):", len(X_test))

Study (train): 1600
Exam (test): 400


In [18]:
demo = ["SWIGGY ORDER", "UBER TRIP", "SWIGGY ORDER AGAIN"]
vectorizer = CountVectorizer()

numbers = vectorizer.fit_transform(demo)

print(vectorizer.get_feature_names_out())
print(numbers.toarray())

['again' 'order' 'swiggy' 'trip' 'uber']
[[0 1 1 0 0]
 [0 0 0 1 1]
 [1 1 1 0 0]]


In [19]:

vectorizer = CountVectorizer()

X_train_numbers = vectorizer.fit_transform(X_train)
X_test_numbers = vectorizer.transform(X_test)

print("Train shape:", X_train_numbers.shape)
print("Test shape:", X_test_numbers.shape)
print("Number of words learned:", len(vectorizer.get_feature_names_out()))

Train shape: (1600, 1477)
Test shape: (400, 1477)
Number of words learned: 1477


In [20]:
vectorizer = CountVectorizer(token_pattern=r"[A-Za-z]+")

X_train_numbers = vectorizer.fit_transform(X_train)
X_test_numbers = vectorizer.transform(X_test)

print("Number of words learned:", len(vectorizer.get_feature_names_out()))
print(vectorizer.get_feature_names_out())

Number of words learned: 40
['airlines' 'airtel' 'amazon' 'bill' 'bookmyshow' 'broadband' 'cabs'
 'chai' 'cinemas' 'dmart' 'dominos' 'flipkart' 'gas' 'hotstar' 'indigo'
 'irctc' 'jio' 'lic' 'metro' 'myntra' 'netflix' 'ola' 'order' 'pay'
 'pizza' 'point' 'power' 'premium' 'pvr' 'recharge' 'reliance' 'spotify'
 'starbucks' 'swiggy' 'ticket' 'torrent' 'trends' 'trip' 'uber' 'zomato']


In [23]:
model = MultinomialNB()
model.fit(X_train_numbers, Y_train)

print("Model trained! ✅")

Model trained! ✅


In [25]:
predictions = model.predict(X_test_numbers)

print("First 5 guesses:", predictions[:5])
print("Real answers:  ", Y_test.values[:5])

print("Accuracy:", accuracy_score(Y_test, predictions))

First 5 guesses: ['Entertainment' 'Shopping' 'Food' 'Travel' 'Travel']
Real answers:   <StringArray>
['Entertainment', 'Shopping', 'Food', 'Travel', 'Travel']
Length: 5, dtype: str
Accuracy: 1.0


In [26]:
my_tests = [
    "SWIGGY ORDER *1234",   # known shop
    "METRO RECHARGE",       # tricky: 'recharge' is in 2 categories
    "JIO RECHARGE",         # tricky: same word, different category
    "RAPIDO BIKE RIDE",     # NEW shop: never seen before!
    "BLINKIT GROCERY",      # NEW shop: never seen before!
]

my_numbers = vectorizer.transform(my_tests)
my_predictions = model.predict(my_numbers)

for text, guess in zip(my_tests, my_predictions):
    print(f"{text:25} → {guess}")

SWIGGY ORDER *1234        → Food
METRO RECHARGE            → Travel
JIO RECHARGE              → Bills
RAPIDO BIKE RIDE          → Entertainment
BLINKIT GROCERY           → Entertainment


In [29]:
probabilities = model.predict_proba(my_numbers)

for text, probs in zip(my_tests, probabilities):
    best = probs.max()
    guess = model.classes_[probs.argmax()]
    print(f"{text:25} → {guess:15} confidence: {best:.0%}")





SWIGGY ORDER *1234        → Food            confidence: 100%
METRO RECHARGE            → Travel          confidence: 98%
JIO RECHARGE              → Bills           confidence: 99%
RAPIDO BIKE RIDE          → Entertainment   confidence: 21%
BLINKIT GROCERY           → Entertainment   confidence: 21%


In [30]:
def categorize(text, threshold=0.6):
    numbers = vectorizer.transform([text])
    probs = model.predict_proba(numbers)[0]
    confidence = probs.max()

    if confidence < threshold:
        return "Uncategorized", confidence
    return model.classes_[probs.argmax()], confidence


for text in my_tests:
    category, conf = categorize(text)
    print(f"{text:25} → {category:15} confidence: {conf:.0%}")

SWIGGY ORDER *1234        → Food            confidence: 100%
METRO RECHARGE            → Travel          confidence: 98%
JIO RECHARGE              → Bills           confidence: 99%
RAPIDO BIKE RIDE          → Uncategorized   confidence: 21%
BLINKIT GROCERY           → Uncategorized   confidence: 21%


In [32]:
models_dir = Path("../backend/models")
models_dir.mkdir(exist_ok=True)

joblib.dump(vectorizer, models_dir / "vectorizer.joblib")
joblib.dump(model, models_dir / "model.joblib")

print("Saved! ✅")

Saved! ✅


In [33]:
loaded_vectorizer = joblib.load(models_dir / "vectorizer.joblib")
loaded_model = joblib.load(models_dir / "model.joblib")

test = loaded_vectorizer.transform(["ZOMATO *9999"])
print(loaded_model.predict(test))

['Food']


## Model Summary
- Model: Naive Bayes with CountVectorizer (letters only, 40 words)
- Test accuracy: 100%, because synthetic data maps each merchant to one category
- Weakness: unseen merchants (e.g. Rapido, Blinkit) get ~21% confidence (random guess)
- Solution: confidence threshold of 60% → "Uncategorized"
- Saved model + vectorizer to backend/models/